# Zynema - Netflix Originals Recommendation


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NetflixOriginals.csv to NetflixOriginals.csv


# **Load Library & Dataset**

In [ ]:
import pandas as pd
import re
import json
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/NetflixOriginals.csv', encoding='latin1')
print(f'Shape: {df.shape[0]} baris, {df.shape[1]} kolom')
df.head(10)

Shape: 584 baris, 6 kolom


,Title,Genre,Premiere,Runtime,IMDB Score,Language
0,Enter the Anime,Documentary,"August 5, 2019",58,2.5,English/Japanese
1,Dark Forces,Thriller,"August 21, 2020",81,2.6,Spanish
2,The App,Science fiction/Drama,"December 26, 2019",79,2.6,Italian
3,The Open House,Horror thriller,"January 19, 2018",94,3.2,English
4,Kaali Khuhi,Mystery,"October 30, 2020",90,3.4,Hindi
5,Drive,Action,"November 1, 2019",147,3.5,Hindi
6,Leyla Everlasting,Comedy,"December 4, 2020",112,3.7,Turkish
7,The Last Days of American Crime,Heist film/Thriller,"June 5, 2020",149,3.7,English
8,Paradox,Musical/Western/Fantasy,"March 23, 2018",73,3.9,English
9,Sardar Ka Grandson,Comedy,"May 18, 2021",139,4.1,Hindi


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Title       584 non-null    object 
 1   Genre       584 non-null    object 
 2   Premiere    584 non-null    object 
 3   Runtime     584 non-null    int64  
 4   IMDB Score  584 non-null    float64
 5   Language    584 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 27.5+ KB


In [17]:
all_genres = []
for g in df['Genre'].dropna():
    genres = [x.strip() for x in g.split(',')]
    all_genres.extend(genres)

unique_genres = sorted(set(all_genres))
print(f'Total genre: {len(unique_genres)}\n')
for i, g in enumerate(unique_genres, 1):
    print(f'  {i:2d}. {g}')

Total genre: 115

   1. Action
   2. Action comedy
   3. Action thriller
   4. Action-adventure
   5. Action-thriller
   6. Action/Comedy
   7. Action/Science fiction
   8. Adventure
   9. Adventure-romance
  10. Adventure/Comedy
  11. Aftershow / Interview
  12. Animated musical comedy
  13. Animation
  14. Animation / Comedy
  15. Animation / Musicial
  16. Animation / Science Fiction
  17. Animation / Short
  18. Animation/Christmas/Comedy/Adventure
  19. Animation/Comedy/Adventure
  20. Animation/Musical/Adventure
  21. Animation/Superhero
  22. Anime / Short
  23. Anime/Fantasy
  24. Anime/Science fiction
  25. Anthology/Dark comedy
  26. Biographical/Comedy
  27. Biopic
  28. Black comedy
  29. Christian musical
  30. Christmas comedy
  31. Christmas musical
  32. Christmas/Fantasy/Adventure/Comedy
  33. Comedy
  34. Comedy / Musical
  35. Comedy horror
  36. Comedy mystery
  37. Comedy-drama
  38. Comedy/Fantasy/Family
  39. Comedy/Horror
  40. Coming-of-age comedy-drama
  41. C

In [30]:
df_clean = df.copy()
df_clean.columns = df_clean.columns.str.strip()

df_clean['Genre'] = df_clean['Genre'].str.lower().str.strip()
df_clean['Title'] = df_clean['Title'].str.lower().str.strip()
df_clean['Language'] = df_clean['Language'].str.lower().str.strip()

df_clean['Genre'] = df_clean['Genre'].str.replace('/', ' ', regex=False)
df_clean['Genre'] = df_clean['Genre'].str.replace('-', ' ', regex=False)
df_clean['Genre'] = df_clean['Genre'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean['Title'] = df_clean['Title'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean['Language'] = df_clean['Language'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Task 1 DONE")
df_clean[['Title', 'Genre', 'Language']].head(3)

Task 1 DONE


,Title,Genre,Language
0,enter the anime,documentary,english/japanese
1,dark forces,thriller,spanish
2,the app,science fiction drama,italian


In [31]:
def map_to_parent_genre(genre_str):
    if pd.isna(genre_str):
        return []
    genre_lower = genre_str.lower()
    parent_mapping = {
        'action':      ['action', 'superhero', 'crime drama', 'thriller', 'heist', 'spy', 'martial'],
        'comedy':      ['comedy', 'dark comedy', 'black comedy', 'satire', 'slapstick', 'parody'],
        'drama':       ['drama', 'biopic', 'historical', 'sports', 'coming of age', 'biographical'],
        'horror':      ['horror', 'zombie', 'supernatural horror', 'psychological horror'],
        'romance':     ['romance', 'romantic', 'love'],
        'animation':   ['animation', 'animated', 'anime', 'stop motion', 'cgi'],
        'scifi':       ['science fiction', 'sci fi'],
        'fantasy':     ['fantasy', 'magical', 'mythology'],
        'family':      ['family', 'christmas', 'holiday', 'kids'],
        'musical':     ['musical', 'concert', 'dance'],
        'documentary': ['documentary', 'making of', 'mockumentary', 'concert film'],
        'mystery':     ['mystery', 'detective'],
        'adventure':   ['adventure', 'exploration'],
        'war':         ['war', 'military'],
        'western':     ['western'],
    }
    matched = []
    for parent, keywords in parent_mapping.items():
        for kw in keywords:
            if kw in genre_lower:
                matched.append(parent)
                break
    return list(set(matched)) if matched else ['other']

df_clean['parent_genre'] = df_clean['Genre'].apply(map_to_parent_genre)
df_clean['parent_genre_str'] = df_clean['parent_genre'].apply(lambda x: ', '.join(x))

print(df_clean['parent_genre_str'].str.split(', ').explode().value_counts())
df_clean[['Title', 'Genre', 'parent_genre_str']].head(5)

Task 2 DONE
parent_genre_str
documentary    169
comedy         143
drama          141
action          93
romance         57
horror          23
animation       22
musical         20
scifi           19
other           15
family          12
adventure       10
fantasy          6
war              5
mystery          4
western          4
Name: count, dtype: int64


,Title,Genre,parent_genre_str
0,enter the anime,documentary,documentary
1,dark forces,thriller,action
2,the app,science fiction drama,"scifi, drama"
3,the open house,horror thriller,"action, horror"
4,kaali khuhi,mystery,mystery


In [35]:
def create_metadata(row):
    title = str(row['Title']).strip() if pd.notna(row['Title']) else 'unknown'
    genre = str(row['parent_genre_str']).strip() if pd.notna(row['parent_genre_str']) else 'unknown'
    language = str(row['Language']).strip() if pd.notna(row['Language']) else 'unknown'
    return f"movie title is {title}. genre is {genre}. language is {language}."

df_clean['metadata'] = df_clean.apply(create_metadata, axis=1)

for m in df_clean['metadata'].head(10):
    print(f"  {m}")

  movie title is enter the anime. genre is documentary. language is english/japanese.
  movie title is dark forces. genre is action. language is spanish.
  movie title is the app. genre is scifi, drama. language is italian.
  movie title is the open house. genre is action, horror. language is english.
  movie title is kaali khuhi. genre is mystery. language is hindi.
  movie title is drive. genre is action. language is hindi.
  movie title is leyla everlasting. genre is comedy. language is turkish.
  movie title is the last days of american crime. genre is action. language is english.
  movie title is paradox. genre is fantasy, western, musical. language is english.
  movie title is sardar ka grandson. genre is comedy. language is hindi.


In [34]:
df_export = df_clean[['Title', 'Genre', 'parent_genre', 'parent_genre_str', 'metadata', 'IMDB Score', 'Runtime', 'Language', 'Premiere']].copy()
df_export.to_csv('netflix_cleaned.csv', index=False)

categories = sorted(df_clean['parent_genre_str'].str.split(', ').explode().unique())
categories_json = [{'id': i+1, 'name': cat} for i, cat in enumerate(categories)]
with open('categories.json', 'w') as f:
    json.dump(categories_json, f, indent=2)

print("Exported: netflix_cleaned.csv")
print("Exported: categories.json")

files.download('netflix_cleaned.csv')
files.download('categories.json')

Exported: netflix_cleaned.csv
Exported: categories.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Poster dari OMDB**




In [ ]:
API_KEY = 'YOUR_API_KEY_HERE' 
def get_poster_omdb(title):
    url = 'https://www.omdbapi.com/'
    params = {'apikey': API_KEY, 't': title}
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if data.get('Response') == 'True':
            poster = data.get('Poster')
            if poster and poster != 'N/A':
                return poster
    return None

posters = []
success = 0
failed = 0

for title in df_clean['Title']:
    poster = get_poster_omdb(title)
    posters.append(poster)
    if poster:
        success += 1
    else:
        failed += 1

df_clean['poster'] = posters

print(f"SUCCESS: {success} | FAILED: {failed}")
print(df_clean[['Title', 'poster']].head(5))

SUCCESS: 551 | FAILED: 33
             Title                                             poster
0  enter the anime  https://m.media-amazon.com/images/M/MV5BNzljM2...
1      dark forces  https://m.media-amazon.com/images/M/MV5BOTIxMW...
2          the app  https://m.media-amazon.com/images/M/MV5BNzgzZG...
3   the open house  https://m.media-amazon.com/images/M/MV5BMTU0Mj...
4      kaali khuhi  https://m.media-amazon.com/images/M/MV5BZjRhNG...


In [ ]:
import requests

def check_poster_url(url):
    if not url:
        return False
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200
    except:
        return False

print("Checking poster URLs...")
valid = 0
broken = 0
broken_info = []

for idx, row in df_clean.iterrows():
    url = row.get('poster')
    if url and check_poster_url(url):
        valid += 1
    else:
        broken += 1
        broken_info.append({'title': row['Title'], 'poster': url})

print(f"VALID: {valid} | BROKEN: {broken}")
print(f"\nBroken posters:")
for b in broken_info:
    print(f"  - {b['title']}")

Checking poster URLs...
VALID: 530 | BROKEN: 54

Broken posters:
  - what happened to mr. cha?
  - 5 star christmas
  - porta dos fundos: the first temptation of christ
  - mrs. serial killer
  - american factory: a conversation with the obamas
  - dolly parton's christmas on the square
  - ghosts of sugar land
  - òlòt?ré
  - the last paradiso
  - choked: paisa bolta hai
  - rodney king
  - frankenstein's monster's monster, frankenstein
  - maska
  - lovefucked
  - tribhanga  tedhi medhi crazy
  - octonauts & the caves of sac actun
  - voyuer
  - june & kopi
  - porta dos fundos: the last hangover
  - layla majnun
  - sturgill simpson presents: sound & fury
  - milestone
  - the american meme
  - counterpunch
  - hope frozen: a quest to live twice
  - a life of speed: the juan manuel fangio story
  - all in my family
  - long live brij mohan
  - have a good trip: adventures in psychedelics
  - bigflo & oil: hip hop frenzy
  - laerte-se
  - remastered: who shot the sheriff?
  - the lo

In [43]:
df_clean['poster_valid'] = df_clean['poster'].apply(lambda x: check_poster_url(x) if x else False)
total_before = len(df_clean)
df_clean = df_clean[df_clean['poster_valid']].drop(columns=['poster_valid']).reset_index(drop=True)
total_after = len(df_clean)
print(f"Total film: {total_after} (dihapus: {total_before - total_after})")

Total film: 528 (dihapus: 56)


In [45]:
df_export = df_clean[['Title', 'Genre', 'parent_genre', 'parent_genre_str', 'metadata', 'poster', 'IMDB Score', 'Runtime', 'Language', 'Premiere']].copy()
df_export.to_csv('netflix_cleaned.csv', index=False)

categories = sorted(df_clean['parent_genre_str'].str.split(', ').explode().unique())
categories_json = [{'id': i+1, 'name': cat} for i, cat in enumerate(categories)]
with open('categories.json', 'w') as f:
    json.dump(categories_json, f, indent=2)

print("Exported: netflix_cleaned.csv")
print("Exported: categories.json")

files.download('netflix_cleaned.csv')
files.download('categories.json')

Exported: netflix_cleaned.csv
Exported: categories.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>